# Diagnostic : améliorer le modèle par les données
## Reconnaissance du genre - noms propres ivoiriens | SIGAP / AFOR
**INP-HB · ESI · Stage 2A · 2025-2026**

Tous les modèles testés (régression logistique, CNN-BiLSTM, v5, boosting) plafonnent vers
92-93 % en validation. Ce notebook teste trois leviers **indépendants de l'algorithme** :

1. **Entrées** : le nom de famille porte-t-il de l'information ? Faut-il séparer NOM et
   PRÉNOMS au lieu de donner le nom complet dans un ordre quelconque ?
2. **Courbe d'apprentissage** : plus de données améliorerait-il le modèle, et de combien ?
3. **Fiabilité** : quelle précision obtient-on en laissant le modèle répondre « incertain »
   sous un seuil de confiance ?

Modèle de mesure : la régression logistique réglée (rapide, meilleur modèle actuel).
**Le jeu de test n'est jamais utilisé.** Aucun nom n'est affiché dans le notebook.

## 0. Paramètres

In [ ]:
RUNS_DIR     = 'runs'
FIGURES_DIR  = 'figures'
MLFLOW_URI   = 'sqlite:///mlflow.db'
N_REPEATS    = 3       # répétitions par taille de la courbe d'apprentissage

## 1. Configuration et données

In [ ]:
import os, re, time, unicodedata, logging, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore', category=UserWarning)
logging.getLogger('mlflow').setLevel(logging.WARNING)
logging.getLogger('alembic').setLevel(logging.WARNING)
os.makedirs(RUNS_DIR, exist_ok=True); os.makedirs(FIGURES_DIR, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('diagnostic')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})


def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    text = re.sub(r"['-]", ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()


df = pd.read_csv('dataset_nettoye.csv')
df['nom']     = df['NOM'].apply(clean_text)
df['prenoms'] = df['PRENOMS'].fillna('').apply(clean_text)
df['full_name'] = (df['nom'] + ' ' + df['prenoms']).str.strip()
df = df[df['full_name'] != ''].reset_index(drop=True)
df['label'] = (df['GENRE'] == 'M').astype(int)

# Découpage FIGÉ : calculé une fois sur les étiquettes d'origine (voir split_reference.csv).
# Le recalculer après une correction d'étiquettes déplacerait des noms entre les ensembles,
# car le découpage est stratifié sur le genre.
SPLIT_PATH = 'split_reference.csv'

def load_split(df):
    import hashlib
    if not os.path.exists(SPLIT_PATH):
        raise FileNotFoundError(f"{SPLIT_PATH} introuvable : il fige le découpage train/val/test")
    sp = pd.read_csv(SPLIT_PATH)
    emp = df['full_name'].map(lambda s: hashlib.sha1(s.encode()).hexdigest()[:10]).values
    assert len(sp) == len(df) and (sp['empreinte'].values == emp).all(), \
        f"{SPLIT_PATH} ne correspond pas au dataset (ordre ou contenu des noms modifié)"
    return [df.index[sp['ensemble'].values == e] for e in ('train', 'val', 'test')]

idx_tr, idx_va, idx_te = load_split(df)
idx_tv = idx_tr.union(idx_va)
tr, va = df.loc[idx_tr], df.loc[idx_va]
y_tr, y_va = tr['label'].values, va['label'].values
print(f"Train={len(tr):,} | Val={len(va):,} | test non utilisé")


def metrics(y, p):
    yp = (p > 0.5).astype(int)
    return dict(acc=float(accuracy_score(y, yp)), f1=float(f1_score(y, yp, average='macro')),
                auc=float(roc_auc_score(y, p)))


def text_features():
    """Représentation de la régression logistique réglée : n-grammes 1-6 + mots."""
    return make_union(TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 6), sublinear_tf=True),
                      TfidfVectorizer(analyzer='word', token_pattern=r'\S+'))

## 2. Expérience 1 : quelles entrées donner au modèle ?

| Variante | Entrée | Correspond à |
|---|---|---|
| `nom_complet` | nom + prénoms en un seul texte | l'app actuelle (un seul champ) |
| `prenoms_seuls` | prénoms uniquement | app à deux champs, nom ignoré |
| `nom_seul` | nom de famille uniquement | mesure l'information portée par le nom |
| `nom_et_prenoms_separes` | deux représentations distinctes | app à deux champs |

`C` est choisi sur la validation pour chaque variante.

In [ ]:
def variant_pipeline(kind, C):
    if kind == 'nom_et_prenoms_separes':
        feats = ColumnTransformer([('prenoms', text_features(), 'prenoms'),
                                   ('nom', text_features(), 'nom')])
    else:
        col = {'nom_complet': 'full_name', 'prenoms_seuls': 'prenoms', 'nom_seul': 'nom'}[kind]
        feats = ColumnTransformer([(col, text_features(), col)])
    return make_pipeline(feats, LogisticRegression(C=C, max_iter=3000))


cols = ['full_name', 'prenoms', 'nom']
exp1, VAL_PROBA = [], {}
for kind in ['nom_complet', 'prenoms_seuls', 'nom_seul', 'nom_et_prenoms_separes']:
    best = None
    for C in [0.5, 1, 2, 3, 5]:
        p = variant_pipeline(kind, C).fit(tr[cols], y_tr).predict_proba(va[cols])[:, 1]
        m = metrics(y_va, p)
        if best is None or m['acc'] > best[0]['acc']:
            best = (m, C, p)
    m, C, p = best
    VAL_PROBA[kind] = p
    exp1.append(dict(variante=kind, C=C, **m))
    with mlflow.start_run(run_name=f'entrees_{kind}'):
        mlflow.log_params(dict(variante=kind, C=C))
        mlflow.log_metrics({f'final_val_{k}': v for k, v in m.items()})

exp1 = pd.DataFrame(exp1)
exp1['ecart_vs_nom_complet'] = exp1['acc'] - exp1.loc[exp1.variante == 'nom_complet', 'acc'].iloc[0]
display(exp1.round(4))

In [ ]:
# Où la variante séparée gagne-t-elle ? Comparaison des erreurs avec le nom complet
from scipy.stats import binomtest
e_full = (VAL_PROBA['nom_complet'] > 0.5) != y_va
for kind in ['prenoms_seuls', 'nom_et_prenoms_separes']:
    e = (VAL_PROBA[kind] > 0.5) != y_va
    b, c = int((e & ~e_full).sum()), int((~e & e_full).sum())
    p = binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
    print(f"{kind:<24} : {c} noms corrigés, {b} nouvelles erreurs | McNemar p = {p:.4f}")

## 3. Expérience 2 : courbe d'apprentissage

Le modèle est entraîné sur 10 % à 100 % du train (sous-échantillons stratifiés, plusieurs
répétitions) et évalué sur la validation. Une courbe encore croissante à 100 % indique que
davantage de données améliorerait le modèle.

In [ ]:
best_kind = exp1.sort_values('acc', ascending=False).iloc[0]
kind, C = best_kind['variante'], best_kind['C']
print(f"Variante mesurée : {kind} (C={C})")

fracs, curve = [0.1, 0.2, 0.35, 0.5, 0.75, 1.0], []
for f in fracs:
    for r in range(N_REPEATS if f < 1 else 1):
        sub = tr if f == 1 else tr.groupby('label', group_keys=False).sample(frac=f, random_state=r)
        p = variant_pipeline(kind, C).fit(sub[cols], sub['label']).predict_proba(va[cols])[:, 1]
        curve.append(dict(fraction=f, n_train=len(sub), acc=metrics(y_va, p)['acc']))
curve = pd.DataFrame(curve).groupby(['fraction', 'n_train'], as_index=False)['acc'].agg(['mean', 'std'])
curve = curve.reset_index() if 'fraction' not in curve.columns else curve
display(curve.round(4))

# Extrapolation simple : acc = a - b * n^(-c), ajustée sur les points mesurés
from scipy.optimize import curve_fit
law = lambda n, a, b, c: a - b * np.power(n, -c)
(a, b, c), _ = curve_fit(law, curve['n_train'], curve['mean'], p0=[0.95, 1.0, 0.3], maxfev=20000)
n_now = len(tr)
for mult in [1, 2, 4]:
    print(f"  {mult}x données ({mult*n_now:,} noms) : acc estimée = {law(mult*n_now, a, b, c):.4f}")
print(f"  plafond estimé (données infinies) : {a:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(curve['n_train'], curve['mean'], yerr=curve['std'].fillna(0), marker='o', capsize=3,
            label='mesuré (validation)')
xs = np.linspace(curve['n_train'].min(), 4 * n_now, 200)
ax.plot(xs, law(xs, a, b, c), '--', label='extrapolation')
ax.axvline(n_now, color='gray', ls=':', label='taille actuelle du train')
ax.set_xlabel("Nombre de noms d'entraînement"); ax.set_ylabel('Accuracy validation')
ax.set_title("Courbe d'apprentissage"); ax.legend()
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/diagnostic_courbe_apprentissage.png', dpi=150)
plt.show()

## 4. Expérience 3 : répondre « incertain » sous un seuil de confiance

Pour chaque seuil, **taux de réponse** (part des noms pour lesquels le modèle répond) et
**précision** sur ces noms. Les autres seraient renvoyés à une vérification humaine.

In [ ]:
rows = []
for kind in ['nom_complet', best_kind['variante']]:
    p = VAL_PROBA[kind]
    conf = np.maximum(p, 1 - p)
    ok = (p > 0.5) == y_va
    for s in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
        keep = conf >= s
        rows.append(dict(variante=kind, seuil=s, taux_reponse=keep.mean(),
                         precision=ok[keep].mean() if keep.any() else np.nan,
                         a_verifier=int((~keep).sum())))
fiab = pd.DataFrame(rows).drop_duplicates(['variante', 'seuil'])
display(fiab.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
for kind, g in fiab.groupby('variante'):
    ax.plot(g['taux_reponse'], g['precision'], marker='o', label=kind)
    for _, r in g.iterrows():
        ax.annotate(f"{r['seuil']:.2f}", (r['taux_reponse'], r['precision']), fontsize=7,
                    xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('Taux de réponse'); ax.set_ylabel('Précision sur les noms traités')
ax.set_title('Compromis précision / taux de réponse'); ax.invert_xaxis(); ax.legend()
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/diagnostic_fiabilite.png', dpi=150)
plt.show()

## 5. Synthèse

Résumé chiffré des trois expériences, enregistré dans `runs/diagnostic_synthese.json`.

In [ ]:
import json
synth = dict(
    entrees=exp1.round(4).to_dict('records'),
    courbe_apprentissage=dict(points=curve.round(4).to_dict('records'),
                              plafond_estime=round(float(a), 4),
                              acc_2x=round(float(law(2*n_now, a, b, c)), 4)),
    fiabilite=fiab.round(4).to_dict('records'))
with open(f'{RUNS_DIR}/diagnostic_synthese.json', 'w') as f:
    json.dump(synth, f, indent=1, ensure_ascii=False)
print(json.dumps({k: v for k, v in synth.items() if k != 'fiabilite'}, indent=1, ensure_ascii=False)[:2000])